<a href="https://colab.research.google.com/github/RodrigoCasanova/Backend/blob/main/Evaluacion_2_Mineria_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EVALUACIÓN 2 Preparacion DE DATOS**

Autores: Rodrigo Casanova, Jose Vasquez, Samuel Acuña

Correo Electrónico: rodr.casanova@duocuc.cl, jo.vasquezp@duocuc.cl, sam.acuna@duocuc.cl

Fecha de Creación: Marzo 2026

Versión: 1.0

In [19]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sb

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from math import log2
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV


# Contexto y Objetivos.

Una empresa de construcción que opera en ciudades como Concepción, Valdivia, Temuco y Punta Arenas busca optimizar la planificación de sus obras considerando las condiciones climáticas.

A partir del análisis de variables como precipitaciones, horas de sol y temperatura, la empresa ajusta sus cronogramas de trabajo para reducir retrasos y mejorar la eficiencia operativa.

# Carga de data


In [20]:
!wget https://raw.githubusercontent.com/RodrigoCasanova/Mineria_de_datos_grupo1/refs/heads/main/datos/data_clima_2025_final.csv

--2026-05-12 03:13:36--  https://raw.githubusercontent.com/RodrigoCasanova/Mineria_de_datos_grupo1/refs/heads/main/datos/data_clima_2025_final.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4527300 (4.3M) [text/plain]
Saving to: ‘data_clima_2025_final.csv’

data_clima_2025_fin 100%[===================>]   4.32M  --.-KB/s    in 0.06s   

2026-05-12 03:13:36 (68.1 MB/s) - ‘data_clima_2025_final.csv’ saved [4527300/4527300]



In [21]:
data = pd.read_csv("data_clima_2025_final.csv", sep=",", low_memory = False)
data.head()

,date,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,cloud_cover,wind_speed_10m,wind_direction_10m,rain,is_day,sunshine_duration,Localidad,latitud,longitud,is_rainy_hour
0,2025-01-01 03:00:00+00:00,15.1,72.053330,13.451475,0.0,0.0,11.609651,187.12492,0.0,0.0,0.0,"Concepción, Chile",-36.82707,-73.050206,0
1,2025-01-01 04:00:00+00:00,14.9,64.437440,12.787600,0.0,0.0,11.341428,179.09064,0.0,0.0,0.0,"Concepción, Chile",-36.82707,-73.050206,0
2,2025-01-01 05:00:00+00:00,14.4,63.023464,12.131762,0.0,0.0,11.032987,174.38250,0.0,0.0,0.0,"Concepción, Chile",-36.82707,-73.050206,0
3,2025-01-01 06:00:00+00:00,13.9,65.991730,11.714258,0.0,0.0,10.805999,178.09090,0.0,0.0,0.0,"Concepción, Chile",-36.82707,-73.050206,0
4,2025-01-01 07:00:00+00:00,13.4,70.770430,11.426943,0.0,0.0,10.299397,185.01303,0.0,0.0,0.0,"Concepción, Chile",-36.82707,-73.050206,0


Detección de nulos

In [22]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33024 entries, 0 to 33023
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  33024 non-null  object 
 1   temperature_2m        33024 non-null  float64
 2   relative_humidity_2m  33024 non-null  float64
 3   apparent_temperature  33024 non-null  float64
 4   precipitation         33024 non-null  float64
 5   cloud_cover           33024 non-null  float64
 6   wind_speed_10m        33024 non-null  float64
 7   wind_direction_10m    33024 non-null  float64
 8   rain                  33024 non-null  float64
 9   is_day                33024 non-null  float64
 10  sunshine_duration     33024 non-null  float64
 11  Localidad             33024 non-null  object 
 12  latitud               33024 non-null  float64
 13  longitud              33024 non-null  float64
 14  is_rainy_hour         33024 non-null  int64  
dtypes: float64(12), int

Detección de valores atípicos

In [23]:
def buscar_atipicos(data: pd.DataFrame, columna: str) -> pd.DataFrame:

    Q1 = data[columna].quantile(0.25)
    Q3 = data[columna].quantile(0.75)

    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    return data[
        (data[columna] < limite_inferior) |
        (data[columna] > limite_superior)
    ]


def obtener_cantidad_atipicos(data: pd.DataFrame, columnas: list) -> dict:

    total_atipicos = {}

    for columna in columnas:
        atipicos = buscar_atipicos(data, columna)
        total_atipicos[columna] = atipicos.shape[0]

    return total_atipicos


columnas_atipicos = [
    'temperature_2m',
    'relative_humidity_2m',
    'apparent_temperature',
    'precipitation',
    'cloud_cover',
    'wind_speed_10m',
    'wind_direction_10m',
    'rain',
    'sunshine_duration'
]


atipicos_por_columna = obtener_cantidad_atipicos(data, columnas_atipicos)


for columna, cantidad in atipicos_por_columna.items():
    print(f"{columna:<30} {cantidad} valores atípicos")

temperature_2m                 476 valores atípicos
relative_humidity_2m           130 valores atípicos
apparent_temperature           282 valores atípicos
precipitation                  6152 valores atípicos
cloud_cover                    0 valores atípicos
wind_speed_10m                 1241 valores atípicos
wind_direction_10m             0 valores atípicos
rain                           6100 valores atípicos
sunshine_duration              0 valores atípicos


Detección de duplicados

In [24]:
# Revisa la existencia de duplicados
data.duplicated().sum()


np.int64(0)

In [25]:
for columna in ["relative_humidity_2m", "precipitation", "cloud_cover","wind_speed_10m","wind_direction_10m","rain","is_day","sunshine_duration","is_rainy_hour"]:
  print(f"{columna : <20} tiene {len(data[data[columna] < 0])} valores negativos")

relative_humidity_2m tiene 0 valores negativos
precipitation        tiene 0 valores negativos
cloud_cover          tiene 0 valores negativos
wind_speed_10m       tiene 0 valores negativos
wind_direction_10m   tiene 0 valores negativos
rain                 tiene 0 valores negativos
is_day               tiene 0 valores negativos
sunshine_duration    tiene 0 valores negativos
is_rainy_hour        tiene 0 valores negativos


In [26]:
print("Humedad fuera de rango:",
      len(data[(data['relative_humidity_2m'] < 0) | (data['relative_humidity_2m'] > 100)]))

print("Dirección viento fuera de rango:",
      len(data[(data['wind_direction_10m'] < 0) | (data['wind_direction_10m'] > 360)]))

Humedad fuera de rango: 0
Dirección viento fuera de rango: 0


# Hora y mes y transformacion ciclica

In [36]:
data["fecha"] = pd.to_datetime(data["date"])
data["hora"] = data["fecha"].dt.hour
data["mes"] = data["fecha"].dt.month

data["hora_sin"] = np.sin(2 * np.pi * data["hora"] / 24)
data["hora_cos"] = np.cos(2 * np.pi * data["hora"] / 24)

data["mes_sin"] = np.sin(2 * np.pi * data["mes"] / 12)
data["mes_cos"] = np.cos(2 * np.pi * data["mes"] / 12)

In [39]:
data = data.sort_values(["Localidad", "fecha"])

data["temp_rolling_6h"] = data.groupby("Localidad")["temperature_2m"].rolling(6).mean().reset_index(0, drop=True)

data["lluvia_rolling_24h"] = data.groupby("Localidad")["precipitation"].rolling(24).sum().reset_index(0, drop=True)

rolling_cols = ["temp_rolling_6h", "lluvia_rolling_24h"]
# Generar los rezagos (Lags) por Localidad
data["temp_lag_1h"] = data.groupby("Localidad")["temperature_2m"].shift(1)
data["temp_lag_2h"] = data.groupby("Localidad")["temperature_2m"].shift(2)
data["temp_lag_3h"] = data.groupby("Localidad")["temperature_2m"].shift(3)
data["temp_lag_24h"] = data.groupby("Localidad")["temperature_2m"].shift(24)
data["temp_lag_12h"] = data.groupby("Localidad")["temperature_2m"].shift(12)
# Es vital eliminar los nuevos nulos generados por el shift
data = data.dropna()
data["nulo_esperado"] = data[rolling_cols].isna().any(axis=1)

Tratamiento de atipicos

In [29]:
class Winsorizer(BaseEstimator, TransformerMixin):
  """
  Tratamiento de atípicos

  Parámetros
  ----------
  BaseEstimator : Clase base para estimadores en scikit-learn.
  TransformerMixin : Clase base para transformadores en scikit-learn.

  Atributos
  ---------
  columns_ : array-like
    Nombres de las columnas a transformar.
  limits : tuple
    % de los extremos a descartar
  """
  def __init__(self, limits=(0.05, 0.05)):
    self.limits = limits

  def fit(self, X, y=None):
    # Guardar nombres si es DataFrame, si no generar nombres genéricos
    if isinstance(X, pd.DataFrame):
      self.columns_ = X.columns
    else:
      self.columns_ = np.arange(X.shape[1])
    return self

  def transform(self, X):
    X = pd.DataFrame(X, columns=self.columns_)
    for col in self.columns_:
      lower = X[col].quantile(self.limits[0])
      upper = X[col].quantile(1 - self.limits[1])
      X[col] = np.clip(X[col], lower, upper)
    return X.values

  def get_feature_names_out(self, input_features=None):
    if input_features is None:
      return np.array(self.columns_)
    else:
      return np.array(input_features)

# Tratamiento de duplicados

In [30]:
def tratar_duplicados(X : pd.DataFrame, drop = True):
  return X.drop_duplicates() if drop else X

Correlacion

In [31]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
  """
  Eliminación de variables correlacionadas

  Parámetros
  ----------
  BaseEstimator : Clase base para estimadores en scikit-learn.
  TransformerMixin : Clase base para transformadores en scikit-learn.

  Atributos
  ---------
  columns_to_drop_ : array-like
    Nombres de las columnas a eliminar.
  threshold : float
    Umbral de correlación.
  Returns
  -------
  DataFrame
    Conjunto de datos sin variables correlacionadas.
  """
  def __init__(self, threshold=0.9):
    self.threshold = threshold
    self.columns_to_drop_ = None

  def fit(self, X, y=None):
    X_df = pd.DataFrame(X)

    corr_matrix = X_df.corr().abs()
    upper = corr_matrix.where(
      np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    self.columns_to_drop_ = [
        col for col in upper.columns if any(upper[col] > self.threshold)
    ]

    return self

  def transform(self, X):
    X_df = pd.DataFrame(X)
    X_filtered = X_df.drop(columns=self.columns_to_drop_, errors="ignore")
    return X_filtered.values

X__train y Test

In [42]:
# División train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=29)

# Entrenamiento
pipeline_lr_temperature.fit(X_train, y_train)

Pipeline(steps=[('duplicados',
                 FunctionTransformer(func=<function tratar_duplicados at 0x7afbf4422e80>,
                                     kw_args={'drop': False})),
                ('preprocesamiento',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('winsorizer',
                                                                   Winsorizer()),
                                                                  ('imputer',
                                                                   SimpleImputer()),
                                                                  ('escalado',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7afbf472f5c0>),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Localidad', 'is_rainy_hour',
                                                   'is_day'])])),
                ('colinealidad', CorrelationFilter()),
                ('modelo', LinearRegression())])

Pipeline

In [41]:
selected_columns = [
    "relative_humidity_2m", "apparent_temperature", "precipitation",
    "cloud_cover", "wind_speed_10m", "wind_direction_10m", "rain",
    "is_day", "sunshine_duration", "Localidad", "latitud", "longitud",
    "is_rainy_hour", "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "temp_rolling_6h", "lluvia_rolling_24h", "temperature_2m","temp_lag_1h","temp_lag_2h","temp_lag_3h","temp_lag_12h","temp_lag_24h"
]

data_for_model = data[selected_columns]
target = "temperature_2m"

X = data_for_model.drop(columns=[target])
y = data[target]

features_num = ["hora_sin", "hora_cos", "mes_sin", "mes_cos", "temp_rolling_6h", "lluvia_rolling_24h",
                "latitud", "longitud", "sunshine_duration","rain", "relative_humidity_2m", "apparent_temperature", "precipitation",
                "cloud_cover", "wind_speed_10m", "wind_direction_10m","temp_lag_1h","temp_lag_2h","temp_lag_24h","temp_lag_12h" ]
features_cat = ["Localidad","is_rainy_hour","is_day", ]
# Preprocesamiento numérico: imputación + transformación
numeric_transformer = Pipeline(steps=[
    ("winsorizer", Winsorizer()),
    ("imputer", SimpleImputer(strategy="mean")),
    ("escalado", StandardScaler())
])

# Preprocesamiento categórico: imputación + one-hot encoding
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

# Combina en un ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, make_column_selector(dtype_include=np.number)),
        ("cat", categorical_transformer, features_cat)
    ],
    remainder="drop"
)

# Crea el pipeline
pipeline_lr_temperature = Pipeline(steps=[
    ("duplicados", FunctionTransformer(tratar_duplicados,
                                       kw_args={"drop": False})),
    ("preprocesamiento", preprocessor),
    ("colinealidad", CorrelationFilter(threshold=0.9)),
    ("modelo", LinearRegression())
])
pipeline_dt_temperature = Pipeline(steps=[
    ("duplicados", FunctionTransformer(tratar_duplicados, kw_args={"drop": False})),
    ("preprocesamiento", preprocessor),
    ("colinealidad", CorrelationFilter(threshold=0.9)),
    ("modelo", DecisionTreeRegressor(max_depth=10, random_state=29))
])

Definimos los pipeline

In [43]:
# 1. Definimos el Pipeline (mantenemos tu estructura ganadora)
pipeline_stress = Pipeline(steps=[
    ("duplicados", FunctionTransformer(tratar_duplicados, kw_args={"drop": False})),
    ("preprocesamiento", preprocessor),
    ("colinealidad", CorrelationFilter(threshold=0.98)), # Umbral alto para procesar más datos
    ("modelo", DecisionTreeRegressor(random_state=29))
])

# 2. El "Grid" de la muerte: Tiramos todos los parámetros posibles
# Esta combinación generará cientos de modelos para entrenar
param_grid_stress = {
    'modelo__max_depth': [10, 15],
    'modelo__min_samples_split': [2],
    'modelo__min_samples_leaf': [1, 4],
    'modelo__criterion': ['squared_error'],
    'modelo__splitter': ['best']
}

# 3. Configuración de GridSearchCV para máximo estrés
# cv=10: Entrenará cada modelo 10 veces (el doble de trabajo)
# n_jobs=-1: Usará el 100% de los núcleos del procesador del Duoc
grid_search_stress = GridSearchCV(
    pipeline_stress,
    param_grid_stress,
    cv=10,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=3 # Para que veas en tiempo real cómo sufre el PC
)

# 4. ¡A correr el estrés!
print("Iniciando prueba de estrés... Prepárate para escuchar los ventiladores.")
grid_search_stress.fit(X_train, y_train)

# 5. Resultados finales[cite: 2]
best_stress_model = grid_search_stress.best_estimator_
y_pred_final = best_stress_model.predict(X_test)

print(f"\n--- RESULTADO DE LA PRUEBA DEFINITIVA ---")
print(f"MAE final: {mean_absolute_error(y_test, y_pred_final):.4f}")
print(f"R² final: {r2_score(y_test, y_pred_final):.4f}")
print(f"Mejores parámetros: {grid_search_stress.best_params_}")

Iniciando prueba de estrés... Prepárate para escuchar los ventiladores.
Fitting 10 folds for each of 4 candidates, totalling 40 fits

--- RESULTADO DE LA PRUEBA DEFINITIVA ---
MAE final: 0.3930
R² final: 0.9830
Mejores parámetros: {'modelo__criterion': 'squared_error', 'modelo__max_depth': 15, 'modelo__min_samples_leaf': 4, 'modelo__min_samples_split': 2, 'modelo__splitter': 'best'}


In [44]:
pipeline_dt_temperature.fit(X_train, y_train)

Pipeline(steps=[('duplicados',
                 FunctionTransformer(func=<function tratar_duplicados at 0x7afbf4422e80>,
                                     kw_args={'drop': False})),
                ('preprocesamiento',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('winsorizer',
                                                                   Winsorizer()),
                                                                  ('imputer',
                                                                   SimpleImputer()),
                                                                  ('escalado',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7afbf472f5c0>),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Localidad', 'is_rainy_hour',
                                                   'is_day'])])),
                ('colinealidad', CorrelationFilter()),
                ('modelo',
                 DecisionTreeRegressor(max_depth=10, random_state=29))])

In [45]:
# Predicciones
y_pred = pipeline_lr_temperature.predict(X_test)

# Evaluación
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- Métricas del modelo Linear regresor---")
print(f"{'R²' :<15}: {r2:.3f}")
print(f"{'MAE':<15}: {mae:,.2f}")


--- Métricas del modelo Linear regresor---
R²             : 0.976
MAE            : 0.40


In [46]:
# Predicciones
y_pred = pipeline_dt_temperature.predict(X_test)

# Evaluación
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- Métricas del modelo Decision Tree Regresor ---")
print(f"{'R²' :<15}: {r2:.3f}")
print(f"{'MAE':<15}: {mae:,.2f}")


--- Métricas del modelo Decision Tree Regresor ---
R²             : 0.979
MAE            : 0.39


In [47]:
# Variables que fueron eliminadas por presentar colinealidad
pipeline_lr_temperature.named_steps["colinealidad"].columns_to_drop_

[6, 10, 16, 18, 19, 20, 23, 26, 27]

In [48]:
X_train.columns

Index(['relative_humidity_2m', 'apparent_temperature', 'precipitation',
       'cloud_cover', 'wind_speed_10m', 'wind_direction_10m', 'rain', 'is_day',
       'sunshine_duration', 'Localidad', 'latitud', 'longitud',
       'is_rainy_hour', 'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos',
       'temp_rolling_6h', 'lluvia_rolling_24h', 'temp_lag_1h', 'temp_lag_2h',
       'temp_lag_3h', 'temp_lag_12h', 'temp_lag_24h'],
      dtype='object')